# 02. Entrenamiento y validación del modelo Random Forest

Este notebook muestra cómo se valida el modelo final de tesis. Para el paper, el modelo oficial es el archivo serializado `models/best_rf_pampa.pkl`.

## Qué debe cambiar el usuario

Para reproducir la tesis, no cambies nada. Si quieres probar otro modelo o nuevos archivos, modifica la celda **Configuración editable**.

In [1]:
from pathlib import Path
import joblib
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score, matthews_corrcoef, recall_score, roc_auc_score

ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
%cd $ROOT

c:\Users\benja\OneDrive - utem.cl\Documentos\Permeability-PAMPA


## Configuración editable

In [2]:
MODEL_FILE = Path('models/best_rf_pampa.pkl')
TRAIN_FILE = Path('data/raw/training_11.csv')
TEST_FILE = Path('data/raw/test_11.csv')
EXTERNAL_FILE = Path('data/raw/external_11.csv')

LABEL_MAP = {'Act-1': 0, 'Act1': 1}
DESCRIPTORS = [
    'LOGPcons', 'MACCSFP125', 'PCR', 'Psi_e_A', 'P_VSA_ppp_D',
    'Mp', 'SpMin1_Bh(p)', 'SHED_AL', 'SM12_AEA(ri)',
    'P_VSA_s_3', 'MATS5m'
]

## Cargar modelo y datos

In [3]:
model = joblib.load(MODEL_FILE)
train = pd.read_csv(TRAIN_FILE)
test = pd.read_csv(TEST_FILE)
external = pd.read_csv(EXTERNAL_FILE)

print('Modelo:', MODEL_FILE)
print('Features del modelo:', list(model.feature_names_in_))

Modelo: models\best_rf_pampa.pkl
Features del modelo: ['LOGPcons', 'MACCSFP125', 'PCR', 'Psi_e_A', 'P_VSA_ppp_D', 'Mp', 'SpMin1_Bh(p)', 'SHED_AL', 'SM12_AEA(ri)', 'P_VSA_s_3', 'MATS5m']


## Evaluar el modelo

La clase positiva es `Act1`, es decir, molécula permeable. El modelo usa `predict` estándar de scikit-learn; no se aplica un umbral manual adicional en esta etapa.

In [4]:
def evaluate(df, name):
    y = df['Actividad'].map(LABEL_MAP).astype(int)
    x = df[DESCRIPTORS]
    pred = model.predict(x).astype(int)
    prob = model.predict_proba(x)[:, 1]
    return {
        'Dataset': name,
        'Accuracy': accuracy_score(y, pred),
        'Sensibilidad': recall_score(y, pred, pos_label=1),
        'Especificidad': recall_score(y, pred, pos_label=0),
        'AUC': roc_auc_score(y, prob),
        'MCC': matthews_corrcoef(y, pred),
        'F1-Score': f1_score(y, pred),
    }

metrics = pd.DataFrame([
    evaluate(train, 'Training'),
    evaluate(test, 'Test interno'),
    evaluate(external, 'Validación externa'),
])
metrics

,Dataset,Accuracy,Sensibilidad,Especificidad,AUC,MCC,F1-Score
0,Training,0.863438,0.852058,0.876543,0.940166,0.727053,0.869775
1,Test interno,0.751376,0.759863,0.741617,0.829928,0.500969,0.765774
2,Validación externa,0.886831,0.939052,0.348837,0.678408,0.290968,0.937993


## Validación automática oficial

Este comando compara las métricas calculadas contra el archivo de resultados guardado en el repositorio. Debe terminar con `[OK]`.

In [5]:
!python src/check_final_metrics.py

Calculated metrics:
                     Accuracy  Sensibilidad  ...    MCC  F1-Score
Dataset                                      ...                 
Training (Original)     0.863         0.852  ...  0.727     0.870
Test Interno            0.751         0.760  ...  0.501     0.766
Validacion Externa      0.887         0.939  ...  0.291     0.938

[3 rows x 6 columns]

Absolute difference vs stored metrics:
                     Accuracy  Sensibilidad  ...       MCC  F1-Score
Dataset                                      ...                    
Training (Original)  0.000438      0.000058  ...  0.000053  0.000225
Test Interno         0.000376      0.000137  ...  0.000031  0.000226
Validacion Externa   0.000169      0.000052  ...  0.000032  0.000007

[3 rows x 6 columns]

[OK] Final metrics match within tolerance=0.001.


## Importancia de variables

Esta tabla muestra qué descriptores usa más el Random Forest. No cambia el modelo; solo ayuda a interpretarlo.

In [6]:
importance = pd.DataFrame({
    'descriptor': DESCRIPTORS,
    'importance': model.feature_importances_,
}).sort_values('importance', ascending=False)
importance

,descriptor,importance
0,LOGPcons,0.257293
2,PCR,0.115743
3,Psi_e_A,0.098618
8,SM12_AEA(ri),0.088549
7,SHED_AL,0.085023
4,P_VSA_ppp_D,0.076562
9,P_VSA_s_3,0.064305
1,MACCSFP125,0.056446
5,Mp,0.055237
6,SpMin1_Bh(p),0.052816


## Salida esperada

Si las métricas coinciden, puedes continuar al notebook `03_Cribado_Virtual_PAMPA.ipynb` para aplicar el modelo a candidatos.